### DATA EXTRACTION AND CLEANING

In [ ]:
import pandas as pd
import numpy as np

# 1. Load and Map Events
# We use these to identify faceoffs and goals specifically
event_map = {
    502: 'faceoff', 503: 'hit', 504: 'giveaway', 505: 'goal', 506: 'shot-on-goal',
    507: 'missed-shot', 508: 'blocked-shot', 509: 'penalty', 516: 'stoppage',
    520: 'period-start', 521: 'period-end', 524: 'game-end', 525: 'takeaway'
}

df = pd.read_csv('Play-By-Play_NHL.csv', low_memory=False)
df['Event'] = df['typeCode'].map(event_map)
df = df.sort_values(['GameID', 'EventIndex']).reset_index(drop=True)

# Helper to determine if the team was on a Powerplay before pulling the goalie
MANPOWER_INV = {
    '5v5': '5v5', '5v4': '4v5', '4v5': '5v4', '4v4': '4v4',
    '3v3': '3v3', '5v3': '3v5', '3v5': '5v3', '4v3': '3v4', '3v4': '4v3'
}

def get_base_strength(df, idx, team):
    game_id = df.at[idx, 'GameID']
    # Look back for the most recent strength state that isn't ENF (Empty Net For) or ENA (Empty Net Against)
    for i in range(idx - 1, -1, -1):
        if i < 0 or df.at[i, 'GameID'] != game_id: break
        state = df.at[i, 'StrengthState']
        event_team = df.at[i, 'EventTeam']
        if pd.notnull(state) and state not in ['ENF', 'ENA']:
            # If the strength state belongs to the pulling team, return it; otherwise, invert it
            return state if event_team == team else MANPOWER_INV.get(state, state)
    return '5v5'

# 2. Identify the Pull Windows (3rd Period only)
pull_events = df[(df['period'] == 3) & (df['StrengthState'] == 'ENF')].copy()

# Capture the context from the very first moment the goalie is pulled
pull_windows = pull_events.groupby(['GameID', 'EventTeam']).agg(
    pull_start_time=('gameTime', 'min'),
    pull_end_time=('gameTime', 'max'),
    Score_When_Pulled=('ScoreState', 'first'),
    Venue=('Venue', 'first'),
    Zone_At_Start=('Zone', 'first'), # Corrected to capital 'Z'
    First_Event_Type=('Event', 'first'),
    First_Row_Index=('GameID', lambda x: x.index[0])
).reset_index()

# Filter for only trailing teams (ScoreState < 0)
pull_windows = pull_windows[pull_windows['Score_When_Pulled'] < 0].copy()

# 3. Add Context: Powerplay Status, Home/Away, and Pull Type
# Powerplay if base strength was 5v4, 5v3, or 4v3
pull_windows['Is_Powerplay_Pull'] = pull_windows.apply(
    lambda x: 1 if get_base_strength(df, x['First_Row_Index'], x['EventTeam']) in ['5v4', '5v3', '4v3'] else 0, axis=1
)

# Pull Type: Before Faceoff (set play) vs During Action (on the fly)
pull_windows['Pull_Occurred_During'] = pull_windows['First_Event_Type'].apply(
    lambda x: 'Before Faceoff' if x == 'faceoff' else 'During Action'
)

# Is Home Team: 1 for Home, 0 for Away (using 'Venue' column)
pull_windows['Is_Home_Team'] = (pull_windows['Venue'] == 'Home').astype(int)

# 4. Scan the full dataset for goals (For and Against) during the pull duration
def get_pull_results(row, full_df):
    # Filter the full game data for the specific window when the net was empty
    window_events = full_df[
        (full_df['GameID'] == row['GameID']) &
        (full_df['gameTime'] >= row['pull_start_time']) &
        (full_df['gameTime'] <= row['pull_end_time'])
    ]

    # Count goals for the pulling team
    goals_for = window_events[(window_events['Event'] == 'goal') &
                              (window_events['EventTeam'] == row['EventTeam'])].shape[0]

    # Count goals against (Empty Net Goals)
    goals_against = window_events[(window_events['Event'] == 'goal') &
                                  (window_events['EventTeam'] != row['EventTeam'])].shape[0]

    return pd.Series([goals_for, goals_against])

pull_windows[['Goals_Scored', 'Empty_Net_Goals_Against']] = pull_windows.apply(
    lambda x: get_pull_results(x, df), axis=1
)

# 5. Final Result Calculations
pull_windows['Seconds_Remaining_At_Pull'] = 3600 - pull_windows['pull_start_time']
pull_windows['Final_Score_Differential'] = pull_windows['Score_When_Pulled'] + pull_windows['Goals_Scored'] - pull_windows['Empty_Net_Goals_Against']
pull_windows['Score_Change'] = pull_windows['Goals_Scored'] - pull_windows['Empty_Net_Goals_Against']

# Was the game tied? (Final differential of 0 or greater)
pull_windows['Did_Successfully_Tie'] = (pull_windows['Final_Score_Differential'] >= 0).astype(int)

# Outcome Classification for risk/reward modeling
# 1: Improved position, -1: Worsened (EN Goal Against), 0: Neutral (No score change)
pull_windows['Outcome_Class'] = 0
pull_windows.loc[pull_windows['Score_Change'] > 0, 'Outcome_Class'] = 1
pull_windows.loc[pull_windows['Score_Change'] < 0, 'Outcome_Class'] = -1

# 6. Final Clean and Export
final_cols = [
    'GameID', 'EventTeam', 'Is_Home_Team', 'Seconds_Remaining_At_Pull',
    'Score_When_Pulled', 'Final_Score_Differential', 'Score_Change',
    'Goals_Scored', 'Empty_Net_Goals_Against', 'Is_Powerplay_Pull',
    'Zone_At_Start', 'Pull_Occurred_During', 'Did_Successfully_Tie', 'Outcome_Class'
]

data = pull_windows[final_cols]
data.to_csv('NHL_Goalie_Pull_Detailed_Analysis.csv', index=False)

print("Detailed Analysis File Created.")

Detailed Analysis File Created.


In [ ]:
data

,GameID,EventTeam,seconds_remaining,start_score,is_home,end_score,score_change,total_goals_scored,opp_goal_allowed,tie_success,risk_reward_class
0,2010020001,MTL,55,-1,0,-1,0,0,0,0,0
2,2010020002,PIT,73,-1,1,-1,0,0,0,0,0
3,2010020003,MIN,87,-1,1,-1,0,0,0,0,0
4,2010020005,CGY,309,-4,0,-4,0,0,0,0,0
5,2010020006,CBJ,83,-1,1,-1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...
11322,2023030412,EDM,347,-2,0,-2,0,0,0,0,0
11323,2023030413,EDM,61,-1,1,-1,0,0,0,0,0
11324,2023030414,FLA,1174,-5,0,-5,0,0,0,0,0
11325,2023030415,FLA,152,-1,1,-1,0,0,0,0,0


In [ ]:
data.to_csv('NHL_Goalie_Pull_Analysis_Data.csv', index=False)